# PUF helper data, key regeneration and AES

These cells use **recorded hardware evidence** by default. They do not program a board.
Use the configured Python (SCU35 PMC) kernel. Fresh hardware commands are shown in Markdown and must be invoked explicitly after configuring your exact SCU35 and recovery image.
Current PL timing uses AXI Timer; native PMC timing uses `mcycle`. JTAG/UART are outside operation timing.

A PUF has no physical Python equivalent. We check registration, volatile key clear, same-helper regeneration and five AES decrypt round trips. No key/helper data is exported. This is not permanent enrollment and does not write eFUSE. Run the shared runtime image explicitly for another board measurement.

In [1]:
from pathlib import Path
import hashlib, json, sys, time
import pandas as pd
# Works from the notebook directory, lab root, or parent repository.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "scu35") if (p / "lab.py").is_file()), None)
assert ROOT is not None, "Launch Jupyter inside this checkout or its scu35 directory"
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "host"))
from compare import parse_capture
from lab import config
capture = parse_capture(ROOT / "evidence/timer/uart.txt", "7d1bf77bc454dd1f")
rows = pd.DataFrame(capture["rows"])
print("Recorded board capture: 31 complete rows; current Python:", sys.version.split()[0])


Recorded board capture: 31 complete rows; current Python: 3.11.9


In [2]:
features = {key: value for key, value in capture["features"].items() if key.startswith("PUF")}
assert {"PUF-REGISTER", "PUF-REGENERATE", "PUF-AES-ENC", "PUF-AES-DEC", "PUF-KEY-CLEAR"}.issubset(features)
display(pd.DataFrame(features).T)
print("PUF lifecycle and same-helper decrypt checks: passed by the strict capture parser")

,repeats,cycles,mean_us,status
PUF-REGISTER,1.0,33715.0,337.15,0.0
PUF-AES-ENC,1.0,4070.0,40.70,0.0
PUF-KEY-CLEAR,5.0,1470.0,2.94,0.0
PUF-REGENERATE,5.0,284105.0,568.21,0.0
PUF-AES-DEC,5.0,17360.0,34.72,0.0


PUF lifecycle and same-helper decrypt checks: passed by the strict capture parser


## Fresh hardware: explicit step

Configure `config.json` first. Run `python lab.py hardware --pdi /path/to/build/demo.pdi` from the lab directory for the full runtime suite, or the native/boot command above for that path. Save logs, verify recovery and compare results before discussing speed. See [audit checkpoints](../docs/audit.md).